In [1]:
source("/home/user/data2/lit/bin/lit_utils.R")
source("/home/user/data3/lit/project/sORFs/sORFs.utils.R")
lib_text()
lib_plot()

In [2]:
fread_c("/home/user/data3/lit/project/sORFs/01-ribo-seq/analysis/20251022_custom_fa_gtf/processed/orfs/PRICE/Human_brain.orfs.tsv") -> orfs

In [4]:
head(orfs)
nrow(orfs)

,Gene,Id,Location,Candidate Location,Codon,Type,Start,Range,p value,/home/user/data3/lit/project/sORFs/01-ribo-seq/analysis/20251022_custom_fa_gtf/processed/merged_bam/,Total
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,WARS2-IT1,PB.2462.1_ncRNA_1,1+:119047264-119047276,1+:119047264-119047276,ATG,ncRNA,0.42,0.27,0.7558600,11.0,11.0
2,WARS2-IT1,PB.2462.1_ncRNA_0,1+:119047334-119047367,1+:119047319-119047367,ATA,ncRNA,0.12,0.20,0.0088559,20.0,20.0
3,WARS2-IT1,PB.2462.1_ncRNA_2,1+:119047477-119047486,1+:119047477-119047486,ATG,ncRNA,0.47,0.24,0.4375000,6.6,6.6
4,PAPPA2,PB.3326.2_ncRNA_0,1+:176555485-176555536,1+:176555485-176555536,AAG,ncRNA,0.42,0.25,0.1162100,14.1,14.1
5,LINC00623,PB.2508.71_ncRNA_1,1+:120942451-120942526,1+:120942451-120942526,AAG,ncRNA,0.13,0.25,0.3567300,44.5,44.5
6,LINC00623,PB.2508.71_ncRNA_2,1+:120952453-120952498,1+:120952453-120952498,ACG,ncRNA,0.47,0.29,0.7388100,32.5,32.5


[1] 193397

In [10]:
orfs %>% filter(`p value`<=0.05) %>% nrow()
orfs %>% filter(`p value`<=0.05 & Codon=="ATG" & Type!="CDS") %>% nrow()

[1] 29197

[1] 5622

In [7]:
adjusted_p <- p.adjust(orfs$`p value`, method = "BH")

In [12]:
sum(adjusted_p<=0.05)
orfs$adjusted_P <- adjusted_p

[1] 18334

In [48]:
head(orfs)
table(orfs$Type)

,Gene,Id,Location,Candidate Location,Codon,Type,Start,Range,p value,/home/user/data3/lit/project/sORFs/01-ribo-seq/analysis/20251022_custom_fa_gtf/processed/merged_bam/,Total,adjusted_P
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,WARS2-IT1,PB.2462.1_ncRNA_1,1+:119047264-119047276,1+:119047264-119047276,ATG,ncRNA,0.42,0.27,0.7558600,11.0,11.0,1.00000000
2,WARS2-IT1,PB.2462.1_ncRNA_0,1+:119047334-119047367,1+:119047319-119047367,ATA,ncRNA,0.12,0.20,0.0088559,20.0,20.0,0.08511698
3,WARS2-IT1,PB.2462.1_ncRNA_2,1+:119047477-119047486,1+:119047477-119047486,ATG,ncRNA,0.47,0.24,0.4375000,6.6,6.6,1.00000000
4,PAPPA2,PB.3326.2_ncRNA_0,1+:176555485-176555536,1+:176555485-176555536,AAG,ncRNA,0.42,0.25,0.1162100,14.1,14.1,0.56824519
5,LINC00623,PB.2508.71_ncRNA_1,1+:120942451-120942526,1+:120942451-120942526,AAG,ncRNA,0.13,0.25,0.3567300,44.5,44.5,0.94337227
6,LINC00623,PB.2508.71_ncRNA_2,1+:120952453-120952498,1+:120952453-120952498,ACG,ncRNA,0.47,0.29,0.7388100,32.5,32.5,1.00000000



    CDS    dORF    iORF   ncRNA  orphan   Trunc   uoORF    uORF Variant 
   2465   28947   23929   98144    2025    8474    5755   20547    3111 

In [33]:
# orfs %>% filter(adjusted_P<=0.05) %>% count(Type)
orfs %>% filter(adjusted_P<=0.05) %>% 
filter(Codon %in% c("ATG","TTG","CTG","GTG","ACG")) %>% 
filter(Type!="Trunc" & Type!="Variant") %>% count(Type) -> tmp
tmp
tmp %>% summarise(sum(n))

Type,n
<chr>,<int>
CDS,1392
dORF,561
iORF,184
ncRNA,3598
orphan,13
uORF,1717
uoORF,675


sum(n)
<int>
8140


In [52]:
orfs %>% filter(adjusted_P<=0.05) %>% 
filter(Codon %in% c("ATG","TTG","CTG","GTG","ACG")) %>% 
filter(Type!="Trunc" & Type!="Variant") %>% count(Codon) %>% mutate(n/sum(n))

Codon,n,n/sum(n)
<chr>,<int>,<dbl>
ACG,580,0.07125307
ATG,3754,0.46117936
CTG,1857,0.22813268
GTG,1205,0.14803440
TTG,744,0.09140049


In [30]:
fread_c("/home/user/data3/lit/project/sORFs/01-ribo-seq/analysis/20251022_custom_fa_gtf/processed/orfs/RiboCode/Human_brain_collapsed.txt") -> ribocode_orfs
head(ribocode_orfs,1) %>% print()

                     ORF_ID ORF_type alt_ORF_type transcript_id transcript_type
1 A1BG_58347580_58347503_25    novel            -    PB.22711.2            None
  gene_id gene_name gene_type chrom start_codon strand ORF_length ORF_tstart
1    A1BG      A1BG      None chr19         ATG      -         75         43
  ORF_tstop ORF_gstart ORF_gstop annotated_tstart annotated_tstop
1       120   58347580  58347503             None            None
  annotated_gstart annotated_gstop Psites_sum_frame0 Psites_sum_frame1
1             None            None                73                12
  Psites_sum_frame2 Psites_coverage_frame0 Psites_coverage_frame1
1                11                 24.00%                 12.00%
  Psites_coverage_frame2 Psites_frame0_RPKM pval_frame0_vs_frame1
1                 12.00%           4.712767            0.02427537
  pval_frame0_vs_frame2 pval_combined adjusted_pval                     AAseq
1            0.05352897   0.005633231    0.01832241 MPSCAARDPSPTSPSSCC

In [55]:
count(ribocode_orfs,ORF_type)
count(ribocode_orfs,start_codon )

ORF_type,n
<chr>,<int>
Overlap_dORF,3250
Overlap_uORF,6810
annotated,20930
dORF,19027
internal,24554
novel,86672
uORF,18708


start_codon,n
<chr>,<int>
ACG,8425
ACg,7
ATG,111238
ATg,213
Acg,6
Atg,74
CTG,27146
CTg,30
Ctg,12
